In [1]:
%store -r

In [2]:
import json
import pathlib
import re

import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session
import pandas

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Load the immuno targets list

Each row has a target name, one or more UniProt IDs (separated by `|` or `;`), and a protein name.

In [4]:
IMMUNO_TARGETS_DIR = RESULTS_DIR / "immuno_targets/"
IMMUNO_TARGETS_FILE = DATA_DIR / "immuno_targets/immuno_targets.csv"
IMMUNO_TARGETS_RESULT_FILE = IMMUNO_TARGETS_DIR / "immuno_targets.json"
IMMUNO_TARGETS_SUMMARY_FILE = IMMUNO_TARGETS_DIR / "immuno_targets_summary.json"
IMMUNO_TARGETS_GRAPHS_DIR = IMMUNO_TARGETS_DIR / "graphs/"
IMMUNO_TARGETS_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
targets_df = pandas.read_csv(IMMUNO_TARGETS_FILE, sep="\t")
targets_df = targets_df.dropna(subset=["UniProtID"]).reset_index(drop=True)
targets_df["UniProtIDs"] = targets_df["UniProtID"].apply(
    lambda raw: [
        uniprot_id.strip()
        for uniprot_id in re.split(r"[|;]", raw)
        if uniprot_id.strip()
    ]
)
targets_df.head()

,Target,UniProtID,ProteinName,UniProtIDs
0,AGER,Q15109,Advanced glycosylation end product-specific re...,[Q15109]
1,AGRP,O00253,Agouti-related protein,[O00253]
2,ANGPT1,Q15389,Angiopoietin-1,[Q15389]
3,ANGPT2,O15123,Angiopoietin-2,[O15123]
4,ANXA1,P04083,Annexin A1,[P04083]


In [6]:
all_uniprot_ids = sorted(
    {
        uniprot_id
        for uniprot_ids in targets_df["UniProtIDs"]
        for uniprot_id in uniprot_ids
    }
)
uniprot_id_to_targets = {}
for _, row in targets_df.iterrows():
    for uniprot_id in row["UniProtIDs"]:
        uniprot_id_to_targets.setdefault(uniprot_id, []).append(row["Target"])
print(f"{len(targets_df)} target rows, {len(all_uniprot_ids)} unique UniProt IDs")

250 target rows, 251 unique UniProt IDs


## Match the immuno targets against proteins in each collection

For every UniProt ID in the immuno targets list, find the proteins in the disease maps that carry an `urn:miriam:uniprot:<id>` RDF annotation, together with their collection, map file path, id, and name.

In [7]:
query = """
WITH $immuno_ids AS immuno_ids
MATCH
    (collection:Collection)-[:HAS_ENTRY]->(collection_entry:CollectionEntry),
    (collection_entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(annotations:Mapping),
    (collection_entry)-[:HAS_OBJ]->(map:CellDesignerMap)-[:HAS_MODEL]->(model:CellDesignerModel),
    (collection_entry)-[:HAS_ID_TO_ELEMENT]->(ids:Mapping),
    (annotations)-[:HAS_ITEM]->(annotations_item:Item),
    (annotations_item)-[:HAS_KEY]->(protein:Protein),
    (annotations_item)-[:HAS_VALUE]->(annotations_bag:Bag),
    (annotations_bag)-[:HAS_ITEM]->(rdf_annotation:RDFAnnotation),
    (ids)-[:HAS_ITEM]->(ids_item:Item),
    (ids_item)-[:HAS_KEY]->(id:String),
    (ids_item)-[:HAS_VALUE]->(protein)
UNWIND rdf_annotation.resources AS resource
WITH immuno_ids, collection, collection_entry, protein, id, resource
WHERE resource STARTS WITH 'urn:miriam:uniprot:'
  AND split(resource, ':')[-1] IN immuno_ids
WITH split(resource, ':')[-1] AS uniprot_id,
     collect(DISTINCT [collection.name, collection_entry.file_path, id.value, protein.name]) AS elements
RETURN uniprot_id, elements
"""
results = session.execute_query(query, params={"immuno_ids": all_uniprot_ids})
elements_by_uniprot_id = {row["uniprot_id"]: row["elements"] for row in results}
print(f"{len(elements_by_uniprot_id)} matched UniProt IDs")

56 matched UniProt IDs


In [8]:
data = []
for _, row in targets_df.iterrows():
    seen = set()
    model_elements = []
    for uniprot_id in row["UniProtIDs"]:
        for element in elements_by_uniprot_id.get(uniprot_id, []):
            collection_name, file_path, model_element_id, model_element_name = element
            key = (collection_name, file_path, model_element_id)
            if key in seen:
                continue
            seen.add(key)
            model_elements.append(
                {
                    "collection": collection_name,
                    "map_file_or_subgraph": file_path,
                    "model_element_id": model_element_id,
                    "model_element_name": model_element_name,
                }
            )
    data.append(
        {
            "target": row["Target"],
            "uniprot_ids": row["UniProtIDs"],
            "protein_name": row["ProteinName"],
            "model_elements": model_elements,
        }
    )
print(
    f"{sum(1 for entry in data if entry['model_elements'])} / {len(data)} targets present in at least one map"
)

55 / 250 targets present in at least one map


## Save results

In [9]:
description = (
    "Immuno targets found in the COVID and PD maps, based on UniProt RDF annotations.\n"
    "This file is generated by the 6_0_analyse_immuno_targets notebook.\n"
    "The list of immuno targets comes from data/immuno_targets/immuno_targets.csv. For each target, "
    "we list every model element (protein) of the COVID or PD maps that carries an "
    "`urn:miriam:uniprot:<id>` RDF annotation matching one of the target's UniProt IDs, together with "
    "the collection it belongs to, the map file path, the model element id, and its name."
)
output = {"description": description, "data": data}
with open(IMMUNO_TARGETS_RESULT_FILE, "w") as f:
    json.dump(output, f)

## Summary

For each collection (and for the intersection of all collections), count how many entries of the immuno targets list have at least one matching model element, and how many model elements (across all maps of the collection) match any entry of the list.

In [10]:
collection_names = sorted(
    {
        model_element["collection"]
        for entry in data
        for model_element in entry["model_elements"]
    }
)

targets_matched_per_collection = {
    collection_name: [
        entry["target"]
        for entry in data
        if any(
            model_element["collection"] == collection_name
            for model_element in entry["model_elements"]
        )
    ]
    for collection_name in collection_names
}
targets_matched_in_all_collections = [
    entry["target"]
    for entry in data
    if entry["model_elements"]
    and all(
        any(
            model_element["collection"] == collection_name
            for model_element in entry["model_elements"]
        )
        for collection_name in collection_names
    )
]
targets_matched_in_any_collection = [
    entry["target"] for entry in data if entry["model_elements"]
]

n_targets_matched = {
    **{name: len(targets) for name, targets in targets_matched_per_collection.items()},
    "any": len(targets_matched_in_any_collection),
    "all": len(targets_matched_in_all_collections),
}
targets_matched = {
    **targets_matched_per_collection,
    "any": targets_matched_in_any_collection,
    "all": targets_matched_in_all_collections,
}

n_model_elements_per_collection = {
    collection_name: sum(
        1
        for entry in data
        for model_element in entry["model_elements"]
        if model_element["collection"] == collection_name
    )
    for collection_name in collection_names
}
n_model_elements_total = sum(len(entry["model_elements"]) for entry in data)

summary = {
    "n_targets": len(data),
    "n_targets_matched": n_targets_matched,
    "targets_matched": targets_matched,
    "n_model_elements_matched": {
        **n_model_elements_per_collection,
        "total": n_model_elements_total,
    },
}

In [11]:
summary_description = (
    "Summary of the immuno targets analysis (see immuno_targets.json for details).\n"
    "`n_targets` is the total number of immuno targets in data/immuno_targets/immuno_targets.csv. "
    "`n_targets_matched` gives, per collection, the number of immuno targets that have at least "
    "one matching model element in that collection; `any` is the number matched in at least one "
    "collection, `all` is the number matched in every collection (the intersection). "
    "`targets_matched` gives, with the same keys, the actual list of target labels behind each count. "
    "`n_model_elements_matched` gives, per collection, the number of model elements that match any "
    "immuno target, and `total` is the sum across collections."
)
summary_output = {"description": summary_description, "data": summary}
with open(IMMUNO_TARGETS_SUMMARY_FILE, "w") as f:
    json.dump(summary_output, f, indent=2)

## Per-map tables

For each collection (PD and COVID), list every map that contains at least one immuno target, together with the targets it contains.

In [12]:
def build_per_map_table(data, collection_name):
    map_to_targets = {}
    for entry in data:
        for model_element in entry["model_elements"]:
            if model_element["collection"] != collection_name:
                continue
            map_name = pathlib.Path(model_element["map_file_or_subgraph"]).name
            map_to_targets.setdefault(map_name, set()).add(entry["target"])
    rows = [
        {"map": map_name, "immuno_targets": ", ".join(sorted(targets))}
        for map_name, targets in sorted(map_to_targets.items())
    ]
    return pandas.DataFrame(rows, columns=["map", "immuno_targets"])

In [13]:
pd_table = build_per_map_table(data, "PD_DM_CD")
with pandas.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(pd_table)

,map,immuno_targets
0,Autophagy.xml,LAMP3
1,Cell_death.xml,GZMB
2,Chaperone_mediated_autophagy.xml,LAMP3
3,Endocytic_vesicles.xml,"LAG3, LAMP3"
4,Histamine_signaling.xml,"IL1B, IL6, TNF"
5,Inflammation_signaling.xml,"IKBKG, IL1B, IL1R1, IRAK4, TNF, TNFRSF1A"
6,Iron_metabolism.xml,"AGER, C1QA, FTH1"
7,MTOR_AMPK_signaling.xml,NAMPT
8,Microglial_phagocytosis.xml,"BDNF, IL1B, MERTK, TREM2"
9,Mitophagy.xml,LAMP3


In [14]:
covid_table = build_per_map_table(data, "COVID_DM_CD")
with pandas.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(covid_table)

,map,immuno_targets
0,Apoptosis_pathway.xml,"FASLG, TNF, TNFRSF1A"
1,Coagulation_pathway.xml,"CRP, CXCL8, IL1B, IL2RA, IL6, KNG1, TNF"
2,HMOX1_pathway.xml,"FTH1, IL18, IL1B"
3,Interferon_1_pathway.xml,"IFNA1; IFNA13, IFNB1, IKBKG, IRAK4, TLR3"
4,Interferon_lambda_pathway.xml,"IFNA1; IFNA13, IFNA2, IFNB1, IFNL1, IFNL2; IFNL3, IL6"
5,Kynurenine_synthesis_pathway.xml,"IFNG, NAMPT"
6,NLRP3_inflammasome_activation.xml,"IL18, IL1B"
7,Nsp14_and_metabolism.xml,NAMPT
8,Orf3a_protein_interactions.xml,"IFNB1, IKBKG, IL1B, TLR3, TNFRSF1A"
9,PAMP_signalling.xml,"IKBKG, IRAK4, TLR3"
